# TFT Model — Training & Prediction using Best Checkpoint

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
from datetime import datetime
from torch.optim.lr_scheduler import ReduceLROnPlateau

from darts.timeseries import TimeSeries
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer
from darts.models import TFTModel

from pytorch_lightning.callbacks import Callback, ModelCheckpoint, EarlyStopping

## 2. Config — Paths & Model Name

In [ ]:
WORK_DIR  = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series"
DATA_PATH = rf"{WORK_DIR}\all_series_data.csv"

# ── Training config ───────────────────────────────────────────────────────────
# MODEL_NAME is stamped with today's date at training time.
# After training, copy the printed MODEL_NAME into LOAD_MODEL_NAME below.
current_date = datetime.now().strftime("%Y-%m-%d")
MODEL_NAME   = f"tft_net_sales_{current_date}"   # e.g. tft_net_sales_2026-05-03

# ── Loading config ────────────────────────────────────────────────────────────
# ⚠️ Set this manually to the MODEL_NAME printed at the end of training.
# Do NOT use MODEL_NAME here — it changes every day when the kernel restarts.
LOAD_MODEL_NAME = "tft_net_sales_2026-05-03"   # ← change this to your actual training date

# Scaler save paths
TARGET_SCALER_PATH      = rf"{WORK_DIR}\scaler_objects\target_scaler.pkl"
FUTURE_SCALER_PATH      = rf"{WORK_DIR}\scaler_objects\future_covariates_scaler.pkl"
STATIC_TRANSFORMER_PATH = rf"{WORK_DIR}\scaler_objects\static_covariate_transformer.pkl"

## 3. Loss Logger Callback

In [ ]:
class LossLogger(Callback):
    """
    Records training and validation losses at the end of every epoch
    for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses   = []

    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get("train_loss")
        if train_loss is not None:
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get("val_loss")
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

## 4. Custom Checkpoint Callback

Subclassing `ModelCheckpoint` to override `state_key` is **required** when using
`save_checkpoints=True` in Darts, because Darts registers its own `ModelCheckpoint`
internally. Without a unique `state_key`, PyTorch Lightning raises:
> RuntimeError: Found more than one stateful callback of type `ModelCheckpoint`.

In [ ]:
class DateStampedCheckpoint(ModelCheckpoint):
    """
    A ModelCheckpoint subclass with a unique state_key so it can coexist
    with Darts's internal ModelCheckpoint without raising a RuntimeError.
    Saves a best checkpoint with the training date, epoch and val_loss
    baked into the filename.
    """
    @property
    def state_key(self) -> str:
        return f"DateStampedCheckpoint_{self.monitor}_{self.dirpath}"

## 5. Add Encoders

In [ ]:
def encode_year(idx):
    """Normalise year relative to year 2000."""
    return (idx.year - 2000) / 50

def encode_days_in_month(index):
    """Return number of days in the month as a feature."""
    return index.days_in_month.to_numpy().reshape(-1, 1)

add_encoders = {
    'cyclic':   {'past': ['month'],    'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom':   {
        'past':   [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

## 6. Read & Prepare Data

In [ ]:
pandas_df = pd.read_csv(DATA_PATH, index_col=['MONTH_OF_SALE'], parse_dates=True)
pandas_df["DEALER_CODE"] = pandas_df["DEALER_CODE"].astype('str')
pandas_df.reset_index().sort_values(
    by=["PARENT_DEALER_CODE_MODEL_FAMILY", "MONTH_OF_SALE"]
).set_index("MONTH_OF_SALE", inplace=True)

In [ ]:
# Identify column types
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

static_cols = pandas_df.select_dtypes(
    exclude=['number', 'datetime', 'datetime64']
).columns.tolist()

# Final column groups
target_col      = ["NET_SALES"]
future_cov_cols = [c for c in target_cols if c != 'NET_SALES']
actual_static   = [c for c in static_cols if c != 'PARENT_DEALER_CODE_MODEL_FAMILY']
static_cov_cols = [
    c for c in actual_static
    if c not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE',
                 'WHEEL_TYPE','BIKE_COLOUR','DEALER_CODE']
]

print(f"Target          : {target_col}")
print(f"Future covariates ({len(future_cov_cols)}): {future_cov_cols}")
print(f"Static covariates: {static_cov_cols}")

In [ ]:
# Separate target+static and future covariate dataframes
df_target_static = pandas_df[['PARENT_DEALER_CODE_MODEL_FAMILY', 'NET_SALES'] + static_cov_cols]
df_future        = pandas_df[future_cov_cols]

## 7. Build Darts TimeSeries Lists

In [ ]:
# ── Build one TimeSeries per series-id ────────────────────────────────────────
# (Replace this block with your own series-building logic if different)

all_series_list            = []
all_future_cov_list        = []

for series_id, group in df_target_static.groupby('PARENT_DEALER_CODE_MODEL_FAMILY'):
    group = group.sort_index()

    ts = TimeSeries.from_dataframe(
        group,
        value_cols=['NET_SALES'],
        static_covariates=group[static_cov_cols].iloc[[0]],
        freq='MS'
    )
    all_series_list.append(ts)

    fut = TimeSeries.from_dataframe(
        df_future.loc[group.index],
        value_cols=future_cov_cols,
        freq='MS'
    )
    all_future_cov_list.append(fut)

print(f"Total series built: {len(all_series_list)}")

## 8. Train / Validation Split

In [ ]:
VAL_MONTHS = 3   # last N months held out for validation

train_list                  = [s[:-VAL_MONTHS] for s in all_series_list]
val_list                    = [s[-VAL_MONTHS:] for s in all_series_list]
train_future_cov_list       = [f[:-VAL_MONTHS] for f in all_future_cov_list]
validation_future_cov_list  = [f[-VAL_MONTHS:] for f in all_future_cov_list]

## 9. Scale Data & Save Scalers

In [ ]:
import os
os.makedirs(rf"{WORK_DIR}\scaler_objects", exist_ok=True)

target_scaler            = Scaler(n_jobs=-1)
future_covariates_scaler = Scaler(n_jobs=-1)
transformer              = StaticCovariatesTransformer(n_jobs=-1)

# Fit & transform training data
scaled_train             = target_scaler.fit_transform(train_list)
scaled_train             = transformer.fit_transform(scaled_train)
scaled_train_fut         = future_covariates_scaler.fit_transform(train_future_cov_list)

# Transform validation data (no re-fitting)
scaled_val               = target_scaler.transform(val_list)
scaled_val               = transformer.transform(scaled_val)
scaled_val_fut           = future_covariates_scaler.transform(validation_future_cov_list)

# Persist scalers
joblib.dump(target_scaler,            TARGET_SCALER_PATH)
joblib.dump(future_covariates_scaler, FUTURE_SCALER_PATH)
joblib.dump(transformer,              STATIC_TRANSFORMER_PATH)

print("Scalers saved.")

## 10. LR Range Test — Find Best Initial Learning Rate

Trains a small TFT for a few epochs across a range of learning rates on a 50-series subset.
Pick the LR with the lowest val_loss as the starting LR for the full training run.
This takes a few minutes but saves many wasted epochs with a bad LR.

In [ ]:
LR_CANDIDATES = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
SUBSET        = 50    # number of series to use for the LR range test

lr_results = {}

for lr in LR_CANDIDATES:
    _logger = LossLogger()
    _model  = TFTModel(
        input_chunk_length=12,
        output_chunk_length=3,
        hidden_size=32,
        lstm_layers=1,
        num_attention_heads=2,
        dropout=0.2,
        batch_size=1024,
        likelihood=None,
        loss_fn=torch.nn.MSELoss(),
        n_epochs=10,                 # just 10 epochs per candidate
        random_state=42,
        add_encoders=add_encoders,
        optimizer_kwargs={'lr': lr, 'weight_decay': 1e-4},
        pl_trainer_kwargs={
            'accelerator'  : 'gpu',
            'precision'    : '16-mixed',
            'callbacks'    : [_logger],
            'enable_checkpointing': False,   # no need to save during LR search
        }
    )
    _model.fit(
        series=scaled_train[:SUBSET],
        future_covariates=scaled_train_fut[:SUBSET],
        val_series=scaled_val[:SUBSET],
        val_future_covariates=scaled_val_fut[:SUBSET],
        verbose=False
    )
    best_val = min(_logger.val_losses) if _logger.val_losses else float('inf')
    lr_results[lr] = best_val
    print(f"LR {lr:.0e}  →  best val_loss: {best_val:.6f}")

BEST_LR = min(lr_results, key=lr_results.get)
print(f"\n✅ Best LR: {BEST_LR:.0e}  (val_loss: {lr_results[BEST_LR]:.6f})")

## 11. Train the Model

Uses `BEST_LR` found above. `ReduceLROnPlateau` will halve it if val_loss plateaus,
and `EarlyStopping` will terminate training if it still does not improve.

In [ ]:
loss_logger = LossLogger()

# ── Early Stopping ────────────────────────────────────────────────────────────
# Stops training when val_loss does not improve for 10 consecutive epochs
# and restores the best weights automatically.
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min',
    verbose=True
)

# ── Dated Best-Checkpoint Saver ───────────────────────────────────────────────
custom_checkpoint_callback = DateStampedCheckpoint(
    monitor='val_loss',
    dirpath=rf"{WORK_DIR}\{MODEL_NAME}\checkpoints",
    filename=f"tft-best-{current_date}-{{epoch:02d}}-{{val_loss:.4f}}",
    save_top_k=1,
    mode='min'
)

# ── TFT Model ─────────────────────────────────────────────────────────────────
# Anti-overfitting settings:
#   dropout=0.2              — regularises attention + feedforward layers
#   hidden_size=32           — smaller model capacity for short (24-36 month) series
#   lstm_layers=1            — reduces sequential overfitting
#   num_attention_heads=2    — sufficient for short series
#   weight_decay=1e-4        — L2 regularisation on all weights
#   BEST_LR                  — tuned by LR range test above
#   ReduceLROnPlateau        — halves LR after 5 epochs of no val_loss improvement
#   EarlyStopping(p=10)      — stops training if LR reduction still cannot help
#   precision=16-mixed       — fp16 mixed precision, maximises GPU throughput
#   batch_size=1024          — large batch for stable gradients and GPU saturation
model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=3,
    hidden_size=32,
    lstm_layers=1,
    num_attention_heads=2,
    dropout=0.2,
    batch_size=1024,
    likelihood=None,
    loss_fn=torch.nn.MSELoss(),
    n_epochs=200,                # high ceiling — early stopping will cut this short
    random_state=42,
    add_encoders=add_encoders,
    optimizer_kwargs={
        'lr'          : BEST_LR,   # from LR range test
        'weight_decay': 1e-4       # L2 regularisation
    },
    lr_scheduler_cls=ReduceLROnPlateau,
    lr_scheduler_kwargs={
        'mode'    : 'min',
        'factor'  : 0.5,           # halve LR on plateau
        'patience': 5,             # plateau = 5 epochs of no improvement
        'min_lr'  : 1e-6,          # never go below this
        'verbose' : True
    },
    model_name=MODEL_NAME,
    work_dir=WORK_DIR,
    save_checkpoints=True,         # saves _model.pth.tar + best-model.ckpt
    force_reset=True,
    pl_trainer_kwargs={
        'accelerator'        : 'gpu',
        'precision'          : '16-mixed',
        'callbacks'          : [loss_logger, early_stopping_callback, custom_checkpoint_callback],
        'enable_checkpointing': True,
    }
)

model.fit(
    series=scaled_train,
    future_covariates=scaled_train_fut,
    val_series=scaled_val,
    val_future_covariates=scaled_val_fut,
    verbose=True
)

print(f"\nModel saved to  : {WORK_DIR}\\{MODEL_NAME}")
print(f"Stopped at epoch: {model.trainer.current_epoch}")
print(f"Best LR used    : {BEST_LR:.0e}")

## 12. Plot Training vs Validation Loss

In [ ]:
def plot_training_results(loss_logger, model):
    trainer    = model.trainer
    optimizer  = trainer.optimizers[0]
    current_lr = optimizer.param_groups[0]["lr"]
    epochs     = range(1, len(loss_logger.train_losses) + 1)

    if len(loss_logger.train_losses) > 0:
        title = (
            f"TFT Training | LR={current_lr:.2e} | Epochs={len(epochs)}\n"
            f"Final Train Loss: {loss_logger.train_losses[-1]:.4f} | "
            f"Final Val Loss: {loss_logger.val_losses[-1]:.4f}"
        )
    else:
        title = "TFT Training (No Data)"

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, loss_logger.train_losses, label="Train Loss",      linewidth=2)
    plt.plot(epochs, loss_logger.val_losses,   label="Validation Loss", linewidth=2, linestyle='--')
    plt.title(title, fontsize=12, fontweight='bold')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(frameon=True)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

plot_training_results(loss_logger, model)

## 12. Load Best Checkpoint for Prediction

Use `MODEL_NAME` and `WORK_DIR` — do **not** pass a direct path to the `.ckpt` file.

In [ ]:
# Uses LOAD_MODEL_NAME — fixed to the date the model was trained.
# This never changes on kernel restart, unlike MODEL_NAME.
loaded_model = TFTModel.load_from_checkpoint(
    model_name=LOAD_MODEL_NAME,
    work_dir=WORK_DIR,
    best=True                # loads Darts' best-model.ckpt automatically
)

print(f"Model loaded successfully: {LOAD_MODEL_NAME}")

## 13. Predict on Validation Set & Export to CSV

In [ ]:
# Load scalers (needed if running in a new session)
target_scaler            = joblib.load(TARGET_SCALER_PATH)
future_covariates_scaler = joblib.load(FUTURE_SCALER_PATH)
transformer              = joblib.load(STATIC_TRANSFORMER_PATH)

In [ ]:
# Generate predictions for the 3 validation months
pred_series = loaded_model.predict(
    n=3,
    series=scaled_train,
    future_covariates=scaled_val_fut
)

# Inverse-transform back to original scale
pred_series_inv = target_scaler.inverse_transform(pred_series)
val_inv         = target_scaler.inverse_transform(scaled_val)

In [ ]:
# Build output DataFrame — one row per series per month
records = []

for actual, forecast, original_series in zip(val_inv, pred_series_inv, val_list):

    # Get the series label from unscaled val_list (retains original string value)
    series_name = original_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = actual.time_index
    actual_values   = actual.values().flatten()
    forecast_values = forecast.values().flatten()

    for month, act, pred in zip(months, actual_values, forecast_values):
        records.append({
            'MONTH_OF_SALE'                  : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'ACTUAL_SALES'                   : round(float(act),  2),
            'PREDICTED_SALES'                : round(float(pred), 2)
        })

df_output = pd.DataFrame(records)
df_output['MONTH_OF_SALE'] = pd.to_datetime(df_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_output = df_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_output.shape}')
print(f'Months       : {df_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_output.head(10)

In [ ]:
# Save to CSV
OUTPUT_CSV = rf"{WORK_DIR}\validation_predictions_{LOAD_MODEL_NAME}.csv"
df_output.to_csv(OUTPUT_CSV, index=False)
print(f'Saved to: {OUTPUT_CSV}')